# 03 — Train heterogeneous R-GCN


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import torch
import torch.nn as nn

from ll_hls4ml.config import load_config
from ll_hls4ml.data.dataset import HeteroGraphDataset
from ll_hls4ml.data.splits import compute_target_stats, random_train_val_split
from ll_hls4ml.models.registry import build
from ll_hls4ml.training import fit, make_loader
from ll_hls4ml.viz.training import plot_loss_curves, plot_predictions_vs_targets


In [ ]:
cfg = load_config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

kernel_types = ["2layer", "exemplar"]
dataset = HeteroGraphDataset(cfg.tensor_dir, types=kernel_types)
train_ds, val_ds = random_train_val_split(dataset, val_fraction=0.2)
y_mean, y_std = compute_target_stats(dataset, train_ds.indices)


In [ ]:
node_vocab_sizes = {nt: 5000 for nt in ["instruction", "variable", "constant"]}

model = build(
    "rgcn",
    node_vocab_sizes=node_vocab_sizes,
    edge_pos_vocab_size=32,
    hidden_dim=128,
    num_layers=5,
).to(device)

train_loader = make_loader(train_ds, batch_size=32, shuffle=True)
val_loader = make_loader(val_ds, batch_size=32, shuffle=False)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
model, history = fit(
    model, train_loader, val_loader,
    epochs=100,
    criterion=nn.MSELoss(),
    optimizer=optimizer,
    device=device,
    y_mean=y_mean,
    y_std=y_std,
    patience=10,
    evaluation_metric="val_loss",
    mode="min",
    experiment_name="cdfg_rgcn_experiment",
    checkpoint_dir=cfg.checkpoint_dir,
)


In [ ]:
plot_loss_curves(history)
plot_predictions_vs_targets(model, val_loader, device, y_mean, y_std)
